# Module 5.4 — Machine Learning Strategies
## Models That Learn—and Markets That Forget Your Assumptions

---

### On Curve-Fitting, Leakage, and the Job Nobody Sees

**Machine learning** in trading is not a competition to maximize **in-sample accuracy**. It is an exercise in **time**: building features that respect **causality**, validating with **purged** or **rolling** schemes that mimic **deployment**, and accepting that **non-stationarity** will eventually break whatever pattern you found Tuesday afternoon. The failure mode is rarely “wrong algorithm”; it is **information leakage** (future bleeding into past), **torturing the test set** (implicitly tuning on holdout), and **mistaking interpolation for edge**.

This notebook is written for people who ship: we engineer **lagged, path-only features**, pose **direction** as **classification**, compare **Random Forest** and **gradient boosting (XGBoost)**—the **tabular** stack most desks still default to for bar-level alpha—score models with **time-series cross-validation**, run a disciplined **walk-forward** backtest on **out-of-sample** predictions only, and stare honestly at **feature importance** and **train–test gaps**. A later sidebar discusses **PyTorch**: when **deep / sequence** models earn their complexity, and why we do not require them here. The philosophical punchline is unchanged from the math: **the optimizer is your enemy** unless you chain it behind **protocol**.

---

### Learning Objectives

1. **Construct** trading **features** from prices alone (lags, rolling stats, ranges) without lookahead
2. **Frame** next-period **direction** as **supervised classification** with clear baseline accuracy
3. **Train** and compare **Random Forest** and **XGBoost** under **time-ordered** validation
4. **Apply** **time series cross-validation** and contrast it with illicit **shuffle** validation (anti-pattern)
5. **Execute** a simple **walk-forward** loop: refit on the past, predict the next segment
6. **Interpret** **feature importance** and **permutation importance** with skepticism
7. **Backtest** an ML-driven rule **only** on **OOS** predictions, with **turnover costs**

---

### Module Contents

1. **Data & baseline** — Synthetic path with weak serial dependence (pedagogy, not a claim on real markets)
2. **Feature engineering** — Lags, volatility, momentum proxies; the label `y`
3. **Time-series CV** — `TimeSeriesSplit`, metrics that do not pretend i.i.d.
4. **Ensembles** — Random Forest vs. XGBoost, calibration to “good enough” regularization
5. **Walk-forward analysis** — Production-shaped retraining cadence
6. **Overfitting diagnostics** — Train vs. validation, sanity checks
7. **OOS backtest & importance** — PnL only where the model was blind; what the model leaned on
8. **Sidebar — PyTorch** — When neural nets are the right tool vs. **overkill**

---

*"If your test set has seen your hyperparameters, it is not a test set—it is a participant in the crime."*


In [1]:
# ========================================
# Setup & reproducible synthetic series
# ========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.inspection import permutation_importance

import xgboost as xgb

from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
plt.rcParams.update({"figure.figsize": (14, 6), "axes.titlesize": 15, "figure.dpi": 100})

COLORS = {
    "price": "#1565C0",
    "ret": "#455A64",
    "rf": "#2E7D32",
    "xgb": "#C62828",
    "oos": "#6A1B9A",
    "bench": "#78909C",
}

print("=" * 78)
print(" " * 10 + "MODULE 5.4: MACHINE LEARNING STRATEGIES")
print("=" * 78)
print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


def simulate_return_series(
    n: int = 3000,
    ar1: float = 0.08,
    vol: float = 0.012,
    seed: int = 42,
) -> pd.DataFrame:
    # Weak AR(1): lets honest lag features carry *some* signal for pedagogy only.
    rng = np.random.default_rng(seed)
    r = np.zeros(n)
    for t in range(1, n):
        r[t] = ar1 * r[t - 1] + vol * rng.standard_normal()
    idx = pd.bdate_range("2015-01-05", periods=n)
    close = 100 * np.exp(np.cumsum(r))
    return pd.DataFrame({"close": close, "ret": pd.Series(r, index=idx)}, index=idx)


df_raw = simulate_return_series()
print(df_raw.head())
print(df_raw.tail())


ModuleNotFoundError: No module named 'sklearn'

---

## 1. Feature Engineering for Trading

### Why quants need this

**Features** are the contract between the world and the model. In live trading, at time $t$ you may use anything **known or realized by $t$** when you act (with care for **reporting lag** in fundamentals). Using $t+1$ data to explain $t$ is **leakage**—the silent Sharpe killer. Quants version-control **feature definitions**, **timezone rules**, and **corporate-action** adjustments because a one-row bug replicates across the entire dataset.

### What we build (price-only)

- **Lagged returns** $r_{t-1}, r_{t-2}, \ldots$
- **Rolling mean / volatility** of returns (causal windows)
- **High–low range proxy** from rolling max/min of close (crude but path-only)

**Label:** $y_t = \mathbb{1}\{r_{t+1} > 0\}$ — prediction uses **only** information available **before** the return being predicted.


In [ ]:
def build_features(price_df: pd.DataFrame, max_lag: int = 8) -> pd.DataFrame:
    out = price_df.copy()
    r = out["ret"]
    for k in range(1, max_lag + 1):
        out[f"r_lag_{k}"] = r.shift(k)
    out["roll_mean_5"] = r.shift(1).rolling(5).mean()
    out["roll_std_20"] = r.shift(1).rolling(20).std()
    out["roll_min_10"] = out["close"].shift(1).rolling(10).min()
    out["roll_max_10"] = out["close"].shift(1).rolling(10).max()
    out["range_10"] = (out["roll_max_10"] - out["roll_min_10"]) / out["close"].shift(1).replace(0, np.nan)
    out["r_fwd"] = r.shift(-1)
    out["y"] = (out["r_fwd"] > 0).astype(int)
    return out


feat = build_features(df_raw)
feat_model = feat.dropna()
feature_cols = [c for c in feat_model.columns if c not in ("close", "ret", "y", "r_fwd")]
X = feat_model[feature_cols]
y = feat_model["y"]

print("Rows after dropna:", len(feat_model))
print("Features:", feature_cols)
print("Label balance (approx P(up)):", y.mean().round(3))
feat_model[["ret", "r_lag_1", "roll_std_20", "y"]].head(10)


---

## 2. Supervised Learning: Classification vs. Naive Baselines

### Why quants need this

**Direction** is a **classification** problem with a **class imbalance** near 50/50 on daily data and **low signal-to-noise**. The baseline is not zero—it is **majority class**, **random**, or **always long** depending on horizon and universe. Quants report **ROC-AUC**, **log loss**, and **economic** metrics (PnL, turnover), not accuracy alone.

### Models

- **Random Forest** — bagged trees; robust, handles nonlinearity; watch **correlated features**
- **XGBoost** — gradient boosting; strong default; tune **depth**, **subsample**, **eta** to curb variance

Both can **overfit** violently if you validate the way kaggle tabular comps validate **i.i.d.** rows—**markets are not i.i.d.**


In [ ]:
def eval_time_series_cv(
    model,
    X: pd.DataFrame,
    y: pd.Series,
    n_splits: int = 5,
) -> pd.DataFrame:
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rows = []
    for fold, (tr, va) in enumerate(tscv.split(X)):
        m = clone_model(model)
        m.fit(X.iloc[tr], y.iloc[tr])
        p = m.predict_proba(X.iloc[va])[:, 1]
        pred_cls = (p >= 0.5).astype(int)
        rows.append(
            {
                "fold": fold,
                "val_acc": accuracy_score(y.iloc[va], pred_cls),
                "val_auc": roc_auc_score(y.iloc[va], p),
                "val_logloss": log_loss(y.iloc[va], p, labels=[0, 1]),
            }
        )
    return pd.DataFrame(rows)


def clone_model(model):
    from sklearn.base import clone
    return clone(model)


rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=50,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=1.0,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    eval_metric="logloss",
)

cv_rf = eval_time_series_cv(rf, X, y)
cv_xgb = eval_time_series_cv(xgb_clf, X, y)
print("RandomForest TimeSeriesSplit folds:")
print(cv_rf.to_string(index=False))
print("\nXGBoost TimeSeriesSplit folds:")
print(cv_xgb.to_string(index=False))
print("\nMean val AUC — RF:", cv_rf["val_auc"].mean().round(4), " XGB:", cv_xgb["val_auc"].mean().round(4))


---

## 3. Time Series Cross-Validation vs. Shuffling (Anti-Pattern)

### Why quants need this

**KFold(shuffle=True)** on financial panels **pastes future folds into training** relative to validation in ways **production never sees**. It **inflates** scores and **mis-ranks** models. **TimeSeriesSplit** is a **minimum** bar; **purged k-fold** (López de Prado) goes further when overlapping labels or event overlap matter.

### Demonstration

We fit the same forest twice: once with **TimeSeriesSplit**, once with **shuffled KFold**. If shuffle looks “better,” that is often **a bug dressed as a result**.


In [ ]:
def eval_shuffle_cv(model, X, y, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rows = []
    for fold, (tr, va) in enumerate(kf.split(X)):
        m = clone_model(model)
        m.fit(X.iloc[tr], y.iloc[tr])
        p = m.predict_proba(X.iloc[va])[:, 1]
        pred_cls = (p >= 0.5).astype(int)
        rows.append(
            {
                "fold": fold,
                "val_acc": accuracy_score(y.iloc[va], pred_cls),
                "val_auc": roc_auc_score(y.iloc[va], p),
            }
        )
    return pd.DataFrame(rows)


base_rf = RandomForestClassifier(
    n_estimators=200, max_depth=6, min_samples_leaf=50, random_state=RANDOM_SEED, n_jobs=-1
)
ts = eval_time_series_cv(base_rf, X, y)
sh = eval_shuffle_cv(base_rf, X, y)
print("Mean AUC — TimeSeriesSplit:", ts["val_auc"].mean().round(4))
print("Mean AUC — Shuffled KFold:  ", sh["val_auc"].mean().round(4))
print("(Shuffled CV is shown as a cautionary contrast, not a recommendation.)")


---

## 4. Ensemble Methods in Production Mindset

### Why quants need this

**Random Forest** averages many **deep but regularized** trees—variance reduction. **XGBoost** fits **sequential** trees to **residuals**—low bias, needs **careful** stopping and **regularization**. In production you also care about **latency**, **serialization**, **feature store contracts**, and **drift monitors**. Here we keep the scope honest: two strong **tabular** baselines, **not** a leaderboard trophy.

### Regularization intuition

Smaller `max_depth`, larger `min_samples_leaf` / `min_child_weight`, **lower** `learning_rate` with **more** trees (early stopping in real pipelines)—all fight **variance** on **non-stationary** data where **true** relationships **move**.


In [ ]:
# Refit on first 70% chronologically, hold out last 30% once for calibration plots
cut = int(len(feat_model) * 0.70)
train_df = feat_model.iloc[:cut]
test_df = feat_model.iloc[cut:]

X_tr, y_tr = train_df[feature_cols], train_df["y"]
X_te, y_te = test_df[feature_cols], test_df["y"]

rf.fit(X_tr, y_tr)
xgb_clf.fit(X_tr, y_tr)

p_rf = rf.predict_proba(X_te)[:, 1]
p_xgb = xgb_clf.predict_proba(X_te)[:, 1]

res = pd.DataFrame(
    {
        "model": ["RandomForest", "XGBoost"],
        "auc": [roc_auc_score(y_te, p_rf), roc_auc_score(y_te, p_xgb)],
        "logloss": [log_loss(y_te, p_rf, labels=[0, 1]), log_loss(y_te, p_xgb, labels=[0, 1])],
    }
)
print("Single chronological split (70/30) — use only for diagnostics, not tuning loops:")
print(res.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(res["model"], res["auc"], color=[COLORS["rf"], COLORS["xgb"]])
ax[0].axhline(0.5, color="gray", ls="--", lw=1)
hi = max(0.55, float(res["auc"].max()) + 0.02)
lo = min(0.45, float(res["auc"].min()) - 0.02)
ax[0].set_ylim(lo, hi)
ax[0].set_title("ROC-AUC (holdout)")
ax[1].bar(res["model"], res["logloss"], color=[COLORS["rf"], COLORS["xgb"]])
ax[1].set_title("Log loss (holdout)")
plt.tight_layout()
plt.show()


---

## 4b. Sidebar — Is PyTorch useful for this chapter?

### Short answer

**Yes, in the right stack—but not as a replacement for what this notebook teaches.** PyTorch (and JAX, etc.) shines when you need **differentiable programs**, **custom architectures**, **GPU-scale** training, or **sequences** where hand-built lags are not enough: **LSTMs/GRUs**, **Transformers** on bars or **tokens** derived from order book state, **representation learning** from high-dimensional microstructure, or **joint** systems (model + execution) trained end-to-end in research settings.

### Why we still center sklearn + XGBoost here

- The README’s core for 5.4 is **tabular ensembles**, **time-ordered validation**, and **walk-forward discipline**—those lessons transfer whether your model is a tree or a neural net.
- **PyTorch does not fix leakage or non-stationarity**; it can **amplify** overfitting if you chase capacity without **protocol**.
- In many production **bar-level** workflows, a **strong baseline** (boosted trees + careful features) is **hard to beat** on engineering ROI; deep models earn their keep when **data scale**, **signal structure**, or **latency/GPU** economics justify them.

### If you extend this notebook

Try a **small** MLP or **1D CNN** on a **fixed-length** tensor of lagged returns with the **same** `TimeSeriesSplit` and **walk-forward** harness—compare **AUC** and **turnover-adjusted** OOS PnL to XGBoost. The win condition is not “neural”; it is **robust OOS behavior** under **real frictions**.


---

## 5. Walk-Forward Analysis

### Why quants need this

**Walk-forward** is how many desks **approximate** reality: **retrain** on a rolling **expanding** or **fixed** history, **predict** the **next** block, **record** OOS performance, **move the window**. It catches **fragility** that a single static split hides. It is **compute-heavy** at scale—hence **feature stores**, **sampling**, and **approximate** retrains.

### Implementation sketch

- **Train** on indices `[0 : t)`
- **Predict** `[t : t + h)`
- Advance `t += h`

Below we use **monthly** steps on business days (approximate): `h ≈ 21` rows.


In [ ]:
def walk_forward_oos_predictions(
    X: pd.DataFrame,
    y: pd.Series,
    r_fwd: pd.Series,
    model_template,
    min_train: int = 400,
    step: int = 21,
) -> pd.DataFrame:
    # Expanding window: each step refit on all past, predict next `step` days.
    # r_fwd[t] = return from t to t+1 (aligns p_up[t] with the PnL bar).
    from sklearn.base import clone

    idx = X.index
    preds = []
    for start in range(min_train, len(X), step):
        end = min(start + step, len(X))
        tr = slice(0, start)
        va = slice(start, end)
        m = clone(model_template)
        m.fit(X.iloc[tr], y.iloc[tr])
        p = m.predict_proba(X.iloc[va])[:, 1]
        block = pd.DataFrame(
            {
                "p_up": p,
                "y": y.iloc[va].values,
                "r_fwd": r_fwd.reindex(idx[va]).values,
            },
            index=idx[va],
        )
        preds.append(block)
    return pd.concat(preds)


r_fwd = feat_model["r_fwd"]
wf_rf = walk_forward_oos_predictions(X, y, r_fwd, rf)
wf_xgb = walk_forward_oos_predictions(X, y, r_fwd, xgb_clf)

print("OOS walk-forward rows — RF:", len(wf_rf), " XGB:", len(wf_xgb))
print("RF OOS AUC:", round(roc_auc_score(wf_rf["y"], wf_rf["p_up"]), 4))
print("XGB OOS AUC:", round(roc_auc_score(wf_xgb["y"], wf_xgb["p_up"]), 4))


---

## 6. Overfitting Prevention: What Actually Matters

### Why quants need this

**High train accuracy + mediocre validation** is the **smoking gun**. So is **thirty variants** tried on the same decade until one “works.” Discipline looks like: **pre-registration** of hypotheses, **nested** CV for tuning, **embargo** when labels overlap, and **paper trading** before capital. No notebook replaces **governance**—this section is the **reminder** taped to the monitor.

### Heuristics (non-exhaustive)

- Prefer **simple** models until simplicity **fails** economically
- Tie complexity to **data size** and **turnover budget**
- Monitor **live** feature distributions vs. **training** (PSI, KS)
- Treat **feature importance** as **debugging**, not **theory**


In [ ]:
rf_full = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=RANDOM_SEED, n_jobs=-1
)
rf_full.fit(X_tr, y_tr)
train_acc = accuracy_score(y_tr, rf_full.predict(X_tr))
test_acc = accuracy_score(y_te, rf_full.predict(X_te))
print(f"More aggressive RF — train acc: {train_acc:.3f}, holdout acc: {test_acc:.3f}")
print("Gap suggests variance; in real work you would not ship on acc alone.")


---

## 7. Feature Importance & ML-Aware Backtesting

### Why quants need this

**Impurity-based importance** favors **high-cardinality** and **correlated** splits; **permutation importance** on a **time-aligned** validation set is often **more honest** for ranking drivers. Neither proves **causality**—markets **regime-shift** and **cancel** yesterday’s story.

### Backtest rule (OOS only)

Use **walk-forward** probabilities: go **long** when `p_up > 0.55`, **flat** otherwise (long-only toy). PnL is **`pos_t × r_fwd[t]`** where `r_fwd[t]` is the **one-step-ahead** return (already stored in `feat_model`); subtract **bps** when `pos` changes. Benchmark = **always long** the same `r_fwd`. This is **illustrative**—not a production execution model.


In [ ]:
def permutation_importance_safe(model, X_va, y_va, feature_cols, seed=42):
    r = permutation_importance(
        model, X_va, y_va, n_repeats=8, random_state=seed, n_jobs=-1, scoring="roc_auc"
    )
    imp = pd.Series(r.importances_mean, index=feature_cols).sort_values(ascending=False)
    return imp


imp_rf = permutation_importance_safe(rf, X_te, y_te, feature_cols)

fig, ax = plt.subplots(figsize=(10, 5))
imp_rf.head(12).iloc[::-1].plot(kind="barh", ax=ax, color=COLORS["rf"])
ax.set_title("Permutation importance (ROC-AUC, holdout) — RandomForest")
plt.tight_layout()
plt.show()

# Native importances (tree bias toward certain features — compare mentally)
nat = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Top native importances:\n", nat.head(8))


def backtest_ml_signal(
    wf: pd.DataFrame,
    threshold: float = 0.55,
    cost_bps: float = 10.0,
) -> pd.DataFrame:
    pos = (wf["p_up"] > threshold).astype(float)
    r_fwd = wf["r_fwd"]
    strat_ret = pos * r_fwd
    turn = pos.diff().abs()
    turn = turn.fillna(pos.abs())
    strat_ret = strat_ret - turn * (cost_bps / 1e4)
    bench = r_fwd
    out = pd.DataFrame({"strat": strat_ret, "bench": bench})
    out["eq_strat"] = (1 + out["strat"]).cumprod()
    out["eq_bench"] = (1 + out["bench"]).cumprod()
    return out


wf_clean = wf_xgb.dropna(subset=["r_fwd"])
bt = backtest_ml_signal(wf_clean, threshold=0.55, cost_bps=12)
print("Walk-forward ML backtest (XGBoost p_up), last equity:", round(float(bt["eq_strat"].iloc[-1]), 4))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(bt.index, bt["eq_strat"], label="ML long/flat (OOS)", color=COLORS["oos"], lw=1.2)
ax.plot(bt.index, bt["eq_bench"], label="Buy & hold returns", color=COLORS["bench"], lw=1, alpha=0.8)
ax.legend()
ax.set_title("Cumulative equity (synthetic data — pedagogical only)")
plt.tight_layout()
plt.show()

print("\n" + "=" * 78)
print("MODULE 5.4 — SUMMARY")
print("=" * 78)
print("Features: causal windows only; label is next return sign.")
print("Validate time-ordered; shuffled CV is for horror stories.")
print("Walk-forward mimics refit cadence; backtest uses OOS preds + costs.")
print("Importance explains the fit, not the future. Ship protocols, not vibes.")
print("=" * 78)
